* ### Transformers libary provides the pre trained models...
* ### The torch library is what actually performs the learning process (backpropagation and weight updates) when you fine-tune a model.
* ### datasets is the data provider & organizer 📦

In [ ]:
!pip install transformers
!pip install torch
!pip install datasets

In [ ]:
from transformers import pipeline

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments


In [ ]:
# ---------------------------
# 1. Training data (chat format)
# ---------------------------
data = [
    {"role": "user", "content": "What is the company name?"},
    {"role": "assistant", "content": "The company name is Intellentx Learning Hub Private Limited."},

    {"role": "user", "content": "What is the CIN of Intellentx Learning Hub Private Limited?"},
    {"role": "assistant", "content": "The CIN is U62099TS2025PTC194300."},

    {"role": "user", "content": "What is the status of the company?"},
    {"role": "assistant", "content": "The company status is Active."},

    {"role": "user", "content": "When was the company incorporated?"},
    {"role": "assistant", "content": "The company was incorporated on 19 February 2025."},

    {"role": "user", "content": "What type of company is Intellentx Learning Hub Private Limited?"},
    {"role": "assistant", "content": "It is a Private Limited Company."},

    {"role": "user", "content": "Which industry does the company belong to?"},
    {"role": "assistant", "content": "The company operates in Business Services and Professional Training."},

    {"role": "user", "content": "Where is the registered office located?"},
    {"role": "assistant", "content": "The registered office is located at Door No. 3-6-111/8/9/T3, Far East Commercial Plaza, Himayatnagar, Telangana 500029, India."},

    {"role": "user", "content": "Who is the registrar of companies for Intellentx Learning Hub Private Limited?"},
    {"role": "assistant", "content": "The company is registered with ROC Hyderabad."},

    {"role": "user", "content": "What is the authorized capital of the company?"},
    {"role": "assistant", "content": "The authorized capital is ₹1.00 Million."},

    {"role": "user", "content": "What is the paid-up capital of the company?"},
    {"role": "assistant", "content": "The paid-up capital is ₹1.00 Million."},

    {"role": "user", "content": "Who are the directors of Intellentx Learning Hub Private Limited?"},
    {"role": "assistant", "content": "The directors are Mohammad Abdul Nayeem, Ahmed Abdul Gaffar, and Naimathullah Khan Mohammed."},

    {"role": "user", "content": "What does Intellentx Learning Hub Private Limited do?"},
    {"role": "assistant", "content": "It is a professional training and learning services company providing industry-oriented tech education in areas such as data science, analytics, cloud computing, cybersecurity, and modern technical skills."},

    {"role": "user", "content": "Who are the typical learners at Intellentx Learning Hub?"},
    {"role": "assistant", "content": "The learners are typically freshers, working professionals, and individuals seeking career enhancement and job-ready skills."},

    {"role": "user", "content": "Is Intellentx Learning Hub known by any other name?"},
    {"role": "assistant", "content": "Yes, it is also known as ITX Learning Hub."},

    {"role": "user", "content": "What courses are offered by Intellentx Learning Hub?"},
    {"role": "assistant", "content": "The company offers courses in Data Analytics, Cloud Computing, Java Full-Stack Development, AI/ML/Deep Learning/Generative AI, and other technology domains."},

    {"role": "user", "content": "What does the Data Analytics course cover?"},
    {"role": "assistant", "content": "The Data Analytics course covers data preparation, data cleaning, visualization techniques, statistical analysis, and reporting insights."},

    {"role": "user", "content": "What is taught in the Cloud Computing course?"},
    {"role": "assistant", "content": "The Cloud Computing course includes training on AWS, Azure, Google Cloud Platform, cloud architecture, deployment models, and hands-on tools."},

    {"role": "user", "content": "What does the Java Full-Stack Development course include?"},
    {"role": "assistant", "content": "The Java Full-Stack Development course includes Java programming, Spring Boot, React, and relational databases such as MySQL."},

    {"role": "user", "content": "What is included in the AI/ML/GenAI Engineer program?"},
    {"role": "assistant", "content": "The program includes Python, Computer Vision using OpenCV, CNNs and YOLO, PyTorch, AI and ML model creation, and cloud AI services."},

    {"role": "user", "content": "Does Intellentx Learning Hub offer bootcamps and workshops?"},
    {"role": "assistant", "content": "Yes, it conducts intensive bootcamps and workshops in Data Science, Full Stack Development, and other technical domains."},

    {"role": "user", "content": "What is the mission of Intellentx Learning Hub?"},
    {"role": "assistant", "content": "The mission is to empower learners with practical expertise, shape careers with technology knowledge, and bridge industry requirements with real skills."},

    {"role": "user", "content": "Where is the training center located?"},
    {"role": "assistant", "content": "The training center is located in Himayat Nagar, Hyderabad, India."},

    {"role": "user", "content": "What services are provided at the training center?"},
    {"role": "assistant", "content": "The training center provides computer training courses, career counseling, flexible schedules, and professional technical education."},

    {"role": "user", "content": "What is the rating of the training institute?"},
    {"role": "assistant", "content": "The institute has an average rating of 4.8 out of 5 based on student reviews."},

    {"role": "user", "content": "What are the operating hours of the training center?"},
    {"role": "assistant", "content": "The training center typically operates from 8 AM to 9 PM."},

    {"role": "user", "content": "What is the contact number of the training center?"},
    {"role": "assistant", "content": "The contact number is 09972815377."},

    {"role": "user", "content": "What is the GSTIN of the company?"},
    {"role": "assistant", "content": "The GSTIN is 36AAICI0394L1ZL."},

    {"role": "user", "content": "Does Intellentx Learning Hub have an online presence?"},
    {"role": "assistant", "content": "Yes, the company is active on social media platforms such as Instagram and professional networks, sharing technical learning content and engaging with the learner community."},

    {"role": "user", "content": "Who are some students studying at Intellentx Learning Hub?"},
    {"role": "assistant", "content": "Some students currently studying include Yethishwar, Rana, Niharika, Yashwanth, Nitish, Raviteja, Jahnavi, Pavan, Omkar, Farhan, Anas, Sadad, and Ikhra in data science and data analytics courses."}
]

df = pd.DataFrame(data)

In [ ]:
def format_chat(data):
    conversations = []
    current = ""

    for msg in data:
        if msg["role"] == "user":
            current += f"User: {msg['content']}\n"
        else:
            current += f"Bot: {msg['content']}\n"
            conversations.append(current.strip())
            current = ""

    return conversations

texts = format_chat(data)
df = pd.DataFrame({"text": texts})
print(df)


In [ ]:
# ---------------------------
# 2. Load model & tokenizer
# ---------------------------
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
# ---------------------------
# 3. Tokenize
# ---------------------------
encodings = tokenizer(
    df["text"].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [ ]:
# ---------------------------
# 4. Dataset
# ---------------------------
class ChatDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = item["input_ids"]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

dataset = ChatDataset(encodings)

In [ ]:
# ---------------------------
# 5. Training arguments
# ---------------------------
training_args = TrainingArguments(
    output_dir="./chatbot",
    num_train_epochs=50,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [ ]:
# ---------------------------
# 6. Train
# ---------------------------
trainer.train()

In [ ]:
# ---------------------------
# 7. Chat function
# ---------------------------
def chat(prompt):
    input_text = f"User: {prompt}\nBot:"
    inputs = tokenizer(input_text, return_tensors="pt")

    # MOVE INPUTS TO SAME DEVICE AS MODEL
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    reply = tokenizer.decode(output[0], skip_special_tokens=True)
    print(reply)

In [ ]:
# ---------------------------
# 8. Test chatbot
# ---------------------------
chat("Hello")
chat("Was the course good?")
chat("Should I take this course?")

In [ ]:
import gradio as gr
import torch

def chat_with_bot(user_input):
    input_text = f"User: {user_input}\nBot:"
    inputs = tokenizer(input_text, return_tensors="pt")

    # move inputs to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    reply = tokenizer.decode(output[0], skip_special_tokens=True)
    return reply

# Gradio UI
interface = gr.Interface(
    fn=chat_with_bot,
    inputs=gr.Textbox(lines=2, placeholder="Type your message here..."),
    outputs="text",
    title="Simple Fine-Tuned Chatbot",
    description="A chatbot fine-tuned using Transformers + Torch"
)

interface.launch()
